<a href="https://colab.research.google.com/github/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/blob/main/_leandro_gapminder_simplificado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise do Desenvolvimento Humano e Econômico Global (Gapminder)

## 1. Título e Introdução

**Grupo:** [nomes]

Breve introdução do tema e objetivos. *(a preencher)*

**Etapas:** 1) Título e Introdução · 2) Coleta, Limpeza e AED · 3) Funções de Visualização · 4) Interpretação · 5) Síntese · 6) Conclusão

## 2. Coleta, Limpeza e Análise Exploratória (AED)

### 2.1 Coleta dos Dados

**#Importar Bibliotecas**

In [52]:
# Importar as bibliotecas que serão utilizadas pelo grupo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from charset_normalizer import from_bytes #Para detectar encoding e separador dos arquivos automaticamente
import urllib.request as ur   #Para aceitar base de dados que venham de URL

**#URLs das bases de Dados**

In [53]:
# Substitua a URL abaixo pela URL 'raw' do seu arquivo no GitHub
url_github =['https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/gapminder_full.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/gasto_saude_per_capita.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/medicos_por_1000_habitantes.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/mortalidade_infantil.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/taxa_obesidade.csv',
              'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/Leandro%20(An%C3%A1lise%20explorat%C3%B3ria%20Sa%C3%BAde)/bases_externas/veiculos_por_1000_habitantes.csv']


**#Detecta o tipo de Encoding e Separador de CSVs**

In [54]:
def configuraçoes_csv(caminho_arquivo):
    """
    Descobre o encoding e o separador de qualquer CSV de forma rápida e certeira.
    Aceita tanto arquivos locais quanto links da internet (ex: GitHub).
    Retorna: (encoding, separador).
    """

    # 1. Verifica se é um link da internet ou arquivo local
    if caminho_arquivo.startswith('http://') or caminho_arquivo.startswith('https://'):   # Verifica se começa com um dos termos
        # Se for link, faz uma requisição HTTP e lê os primeiros 20.000 bytes
        req = ur.Request(
            caminho_arquivo,
            headers={'User-Agent': 'Mozilla/5.0'} # Simula um cabeçalho real na requisição como se fosse um usuário acessando pelo Mozila
        )
        with ur.urlopen(req) as f:   # Abre a conexão com o servidor através da requisição anterior criada
            amostra_binaria = f.read(20000)   #Lê e armazena os primeiros 20000 bytes em amostra_binária
                                              #NOTA: não usamos open(caminho_arquivo, "rb") pq na internet tudo já está em byte
    else:
        # Se for arquivo local, abre normalmente
        with open(caminho_arquivo, 'rb') as f:    #Aqui usamos o 'rb' pois arquivos locais não costuman estar em binário pois nós não conseguiriamos ler.
            amostra_binaria = f.read(20000)

    # 2. Descobre o encoding com quase 100% de precisão
    resultado = from_bytes(amostra_binaria).best()  # Analisa a amostra binária
    encoding = resultado.encoding   # Armazena o nome do encoding detectado
    texto_amostra = str(resultado)    # Converte em texto legível novamente

    # 3. Descobre o separador (.csv, .txt, etc)
    dialeto = csv.Sniffer().sniff(texto_amostra)    #"Fareja" em nossa amostra qual o separador mais aparece e
    separador = dialeto.delimiter

    return caminho_arquivo,encoding, separador

**#Cria um único DataFrame**

In [55]:
def cria_dataframe(caminho_arquivo,encoding,separador):
    df=pd.read_csv(caminho_arquivo, encoding=encoding, sep=separador)
    return df


**#Cria múltiplos DataFrames recebendo lista de URLs ou CSVs**

In [56]:
def carrega_multiplos_dataframes(lista_caminhos):
    """
    Recebe uma lista de URLs ou caminhos locais e retorna uma lista de DataFrames carregados
    automaticamente usando as configurações detectadas para cada um deles.

    **Exemplo de entrada:**
    lista caminhos= [url1, url2, url3, ...] ou [caminho1, caminho2, caminho3, ...]

    **Exemplo de saída:**
    dataframes= [df1, df2, df3, ...]
    """
    dataframes = []   #cria uma lista vazia para nosso append ir add oq precisa
    for caminho in lista_caminhos:
        # 1. Detecta as configurações do arquivo atual
        config = configuraçoes_csv(caminho)
        # 2. Cria o DataFrame correspondente
        df = cria_dataframe(*config)#garante que cada termo da tupla seja descompactado e utilizado na função
        # 3. Adiciona o DataFrame à lista
        dataframes.append(df)
    return dataframes

# Exemplo de uso prático:
# lista_de_fontes = [url_github]  # Você pode colocar quantos links ou caminhos quiser aqui
# meus_dfs = carregar_multiplos_dataframes(lista_de_fontes)
# meus_dfs[0].head()  # Acessa o primeiro DataFrame gerado

**#Nomeia cada DataFrame**

### 2.2 Limpeza dos Dados


**#Faz uma cópia dos dados brutos para tratamento e limpeza**

In [67]:
dfs_brutos = carrega_multiplos_dataframes(url_github).copy()

**#Detecta NULL,NA,e duplicados**

In [58]:
def remove_duplicados_e_vazios(df_bruto):
    df = df_bruto.copy()

    # Conta a quantidade de LINHAS que contêm pelo menos um valor nulo/vazio ou duplicados
    qtd_linhas_null = df.isnull().any(axis=1).sum()
    qtd_linhas_vazias = df.isna().any(axis=1).sum()
    qtd_linhas_duplicadas = df.duplicated().sum()

    print(f"""A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: {qtd_linhas_null}
              Linhas com celulas Vazias: {qtd_linhas_vazias}
              Linhas Duplicadas: {qtd_linhas_duplicadas}""")

**#Informa situação de linhas de cada DataFrame (NULLs,NAs e duplicados)**

In [59]:
remove_duplicados_e_vazios(df_pib)
remove_duplicados_e_vazios(df_investimento_saude)
remove_duplicados_e_vazios(df_investimento_medicos)
remove_duplicados_e_vazios(df_mortalidade_infantil)

A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: 0
              Linhas com celulas Vazias: 0
              Linhas Duplicadas: 32
A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: 0
              Linhas com celulas Vazias: 0
              Linhas Duplicadas: 0
A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: 0
              Linhas com celulas Vazias: 0
              Linhas Duplicadas: 0
A quantidade de itens excluidos se dividem em:
              Linhas com celulas Nulas: 0
              Linhas com celulas Vazias: 0
              Linhas Duplicadas: 0


**#Apresenta a estrutura de Cada Data Frame**

In [60]:
df_pib.describe()


,year,pop,lifeExp,gdpPercap
count,1736.000000,1.736000e+03,1736.000000,1736.000000
mean,1979.669931,3.156800e+07,59.511979,7240.155986
std,17.296364,1.172888e+08,12.956028,9839.873695
min,1952.000000,6.001100e+04,23.599000,241.165876
25%,1967.000000,2.818694e+06,48.245000,1202.060309
50%,1982.000000,7.150606e+06,60.808000,3534.772106
75%,1997.000000,1.977102e+07,70.945000,9380.254270
max,2007.000000,1.318683e+09,82.603000,113523.132900


In [61]:
df_investimento_saude.describe()

,Year,"Current health expenditure per capita, PPP (current international $)"
count,4874.000000,4874.000000
mean,2011.652236,1323.041766
std,6.953682,1790.876001
min,2000.000000,6.106393
25%,2006.000000,176.241105
50%,2012.000000,599.093750
75%,2018.000000,1701.851275
max,2024.000000,13473.193000


In [62]:
df_investimento_medicos.describe()

,Year,"Physicians (per 1,000 people)"
count,5405.000000,5405.000000
mean,1998.487697,1.729846
std,17.051683,1.463668
min,1960.000000,0.004000
25%,1987.000000,0.388000
50%,2001.000000,1.435000
75%,2013.000000,2.828000
max,2023.000000,9.542000


In [63]:
df_mortalidade_infantil.describe()

,Year,Under-five mortality rate (selected)
count,17066.000000,17066.000000
mean,1975.053557,10.656250
std,41.387351,10.781884
min,1751.000000,0.130000
25%,1960.000000,2.130000
50%,1984.000000,6.490000
75%,2004.000000,16.620000
max,2024.000000,78.090000


In [64]:
df_taxa_obesidade.describe()

,Year,"Obesity among adults, BMI >= 30 kg/m2 (crude estimate) (%) - Sex: both sexes - Age group: 18+ years of age"
count,9270.000000,9270.000000
mean,2002.000000,15.838194
std,12.987874,13.223721
min,1980.000000,0.124788
25%,1991.000000,5.891188
50%,2002.000000,13.292583
75%,2013.000000,22.290522
max,2024.000000,75.812290


### 2.3 Análise Exploratória (AED)
*(a fazer: mínimo 15 análises)*

## 3. Funções de Visualização e Análise Gráfica
*(a fazer: função reutilizável de gráfico)*

## 4. Interpretação das Visualizações
*(a preencher)*

## 5. Síntese dos Principais Insights
*(a preencher)*

## 6. Conclusão e Reflexão
*(a preencher)*